# Introduction to ChromaDB: Product Search Demo

This notebook is inspired by the beginner ChromaDB tutorial and demonstrates a more practical use case: a product catalog search system powered by semantic similarity.

By the end of this notebook, you will learn:
1. How to create a ChromaDB client
2. How to store product information in a collection
3. How to search by meaning instead of exact keyword match
4. How to filter results using metadata
5. How to update or delete documents


---

## 1. Install the required packages

We use ChromaDB to store and query embeddings, and we also install `sentence-transformers` to generate embeddings for text.

In [ ]:
!python -m pip install -q chromadb sentence-transformers

---

## 2. Import libraries and create a persistent database

A persistent database is useful because it saves the vector data on disk and keeps it available across notebook runs.

In [ ]:
import chromadb
from pathlib import Path

persist_dir = Path("./chroma_product_db").resolve()
persist_dir.mkdir(exist_ok=True)

client = chromadb.PersistentClient(path=str(persist_dir))

print("✓ ChromaDB client created successfully!")
print(f"Storage location: {persist_dir}")

---

## 3. Create a collection for product catalog search

A collection is like a table in a database. Here we store product descriptions and their semantic embeddings.

In [ ]:
collection = client.get_or_create_collection(
    name="product_catalog",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection created: {collection.name}")
print(f"Current count: {collection.count()}")

---

## 4. Add products to the vector store

Each product gets an id, metadata, and a description. ChromaDB converts these texts into embeddings automatically when you add them.

In [ ]:
products = [
    {
        "id": "p1",
        "name": "Wireless Noise-Canceling Headphones",
        "category": "electronics",
        "brand": "AudioMax",
        "description": "Premium over-ear headphones with active noise cancellation, immersive bass, and 30-hour battery life."
    },
    {
        "id": "p2",
        "name": "Smart Fitness Watch",
        "category": "electronics",
        "brand": "FitPro",
        "description": "A lightweight smartwatch that tracks heart rate, sleep, workouts, and notifications."
    },
    {
        "id": "p3",
        "name": "Organic Green Tea",
        "category": "grocery",
        "brand": "NatureLeaf",
        "description": "Organic green tea rich in antioxidants and perfect for a healthy daily routine."
    },
    {
        "id": "p4",
        "name": "Running Shoes",
        "category": "sports",
        "brand": "SprintZone",
        "description": "Lightweight running shoes with responsive cushioning for everyday workouts and marathons."
    },
    {
        "id": "p5",
        "name": "4K Smart TV",
        "category": "electronics",
        "brand": "VisionPlus",
        "description": "Ultra HD television with HDR support, voice assistant, and access to streaming apps."
    },
    {
        "id": "p6",
        "name": "Laptop Stand",
        "category": "office",
        "brand": "DeskEase",
        "description": "Ergonomic aluminum laptop stand that improves posture and keeps devices cool."
    },
    {
        "id": "p7",
        "name": "Protein Powder",
        "category": "health",
        "brand": "FuelFit",
        "description": "High-protein whey supplement designed for muscle recovery and daily energy support."
    },
    {
        "id": "p8",
        "name": "Cotton T-Shirt",
        "category": "fashion",
        "brand": "UrbanWear",
        "description": "Soft everyday cotton T-shirt made for casual comfort and style."
    }
]

documents = []
metadatas = []
ids = []

for product in products:
    text = f"{product['name']} | {product['category']} | {product['brand']} | {product['description']}"
    documents.append(text)
    metadatas.append({
        "category": product["category"],
        "brand": product["brand"]
    })
    ids.append(product["id"])

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("✓ Products added successfully!")
print(f"Total documents in collection: {collection.count()}")

---

## 5. Search by semantic similarity

This is where vector search becomes powerful. A query like "device for listening to music without noise" can match a product such as wireless headphones even if the exact words are different.

In [ ]:
query = "I want a device for listening to music without noise from outside"

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Query: {query}")
print("\nTop matching products:\n")

for doc, metadata, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    similarity = 1 - distance
    print(f"Document: {doc}")
    print(f"Category: {metadata.get('category')}")
    print(f"Brand: {metadata.get('brand')}")
    print(f"Distance: {distance:.4f} | Similarity: {similarity:.4f}")
    print()

---

## 6. Filter results by metadata

Metadata allows us to narrow down searches by category, brand, or other labels. For example, we can search only within electronics products.

In [ ]:
filter_query = "I need something to help with home workouts and tracking health"

results = collection.query(
    query_texts=[filter_query],
    n_results=3,
    where={"category": "electronics"}
)

print(f"Query: {filter_query}")
print("\nFiltered results (only electronics):\n")

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | {doc}")

---

## 7. Update a product

Sometimes product information changes. ChromaDB lets us update the document content for an existing ID.

In [ ]:
collection.update(
    ids=["p4"],
    documents=[
        "SprintZone running shoes with improved cushioning, breathable mesh, and extra support for jogging and training."
    ],
    metadatas=[{"category": "sports", "brand": "SprintZone"}]
)

print("✓ Product p4 updated successfully!")

---

## 8. Delete a product

We can also remove documents that are no longer relevant.

In [ ]:
collection.delete(ids=["p3"])

print("✓ Product p3 deleted.")
print(f"Current count: {collection.count()}")

---

## 9. Key takeaways

✅ ChromaDB is useful because it helps us search by meaning, not just exact words.
✅ Metadata like category and brand can be used to reduce search noise.
✅ Vector databases support real-world AI features such as product search, recommendations, and retrieval systems.

This product search example is a simplified version of how AI-powered stores and search systems work in practice.

In [ ]:
print("\n" + "=" * 55)
print("CHROMADB PRODUCT SEARCH DEMO COMPLETE")
print("=" * 55)
print(f"Collection: {collection.name}")
print(f"Final document count: {collection.count()}")
print("You now understand the basics of ChromaDB for semantic product search!")